In [ ]:
"""
Traffic Demand Forecasting — Solution v6
OOF R2: 96.59  |  Online score: 91.85534

Pipeline:
  1. Feature engineering (geo decode, time features, lag features)
  2. K-Fold target encoding
  3. 3-seed HistGradientBoosting (HGBT) ensemble
  4. 2-seed ExtraTrees (ET) ensemble
  5. 3-way OOF-optimised blend: HGBT×0.55 + ET_seed42×0.00 + ET_seed2024×0.45
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# CONFIG — adjust paths as needed
# ─────────────────────────────────────────────────────────────
TRAIN_PATH  = '../data/dataset/train.csv'
TEST_PATH   = '../data/dataset/test.csv'
OUTPUT_PATH = '../submissions/submission_v6.csv'

# ─────────────────────────────────────────────────────────────
# 1. GEOHASH DECODE  (no pygeohash required)
# ─────────────────────────────────────────────────────────────
BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'

def decode_geohash(geohash):
    lat_range, lon_range = [-90.0, 90.0], [-180.0, 180.0]
    is_lon = True
    for char in geohash:
        bits = BASE32.index(char)
        for i in range(4, -1, -1):
            bit = (bits >> i) & 1
            if is_lon:
                mid = (lon_range[0] + lon_range[1]) / 2
                lon_range[bit] = mid
            else:
                mid = (lat_range[0] + lat_range[1]) / 2
                lat_range[bit] = mid
            is_lon = not is_lon
    return (lat_range[0] + lat_range[1]) / 2, (lon_range[0] + lon_range[1]) / 2

# ─────────────────────────────────────────────────────────────
# 2. LOAD
# ─────────────────────────────────────────────────────────────
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"  Train: {train_df.shape}, Test: {test_df.shape}")

# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("Feature engineering...")
geo_cache = {g: decode_geohash(g)
             for g in set(train_df.geohash) | set(test_df.geohash)}

for df in [train_df, test_df]:
    df['latitude']  = df['geohash'].map(lambda g: geo_cache[g][0])
    df['longitude'] = df['geohash'].map(lambda g: geo_cache[g][1])

    ts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'], df['minute'] = ts[0], ts[1]
    df['time_slot']  = df['hour'] * 4 + df['minute'] // 15

    df['slot_sin']   = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos']   = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)

    df['is_weekend'] = (df['day'] % 7).isin([0, 6]).astype(int)
    df['is_rush']    = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df['is_night']   = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)

    df['geo3']       = df['geohash'].str[:3]
    df['geo4']       = df['geohash'].str[:4]
    df['geo5']       = df['geohash'].str[:5]

    df['road_hour']    = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)
    df['geo_slot']     = df['geohash'] + '_' + df['time_slot'].astype(str)
    df['geo_hour']     = df['geohash'] + '_' + df['hour'].astype(str)
    df['road_slot']    = df['RoadType'].astype(str) + '_' + df['time_slot'].astype(str)
    df['geo5_slot']    = df['geo5'] + '_' + df['time_slot'].astype(str)
    df['lanes_x_slot'] = df['NumberofLanes'] * df['time_slot']

    df['Temperature'] = df.groupby('geohash')['Temperature'].transform(
        lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

    for c in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        df[c] = df[c].fillna('Unknown')

# ─────────────────────────────────────────────────────────────
# 4. LAG FEATURES  (fully vectorised — no row iteration)
# ─────────────────────────────────────────────────────────────
print("Building lag features...")
train48 = train_df[train_df.day == 48].copy()
train49 = train_df[train_df.day == 49].copy()

d48_exact_ts   = train48.groupby(['geohash', 'timestamp'])['demand'].mean()
d48_exact_slot = train48.groupby(['geohash', 'time_slot'])['demand'].mean()
slot_med       = train_df.groupby('time_slot')['demand'].median()
geo_mean_d48   = train48.groupby('geohash')['demand'].mean()


def add_lag(df):
    """Build demand_lag_d1: exact d48 timestamp → adjacent slots → geo mean → slot median."""
    d = df[['geohash', 'timestamp', 'time_slot']].copy().reset_index(drop=True)
    # 1. Exact (geohash, timestamp) match from day48
    d = d.merge(d48_exact_ts.rename('lag').reset_index(),
                on=['geohash', 'timestamp'], how='left')
    # 2. Adjacent slots (±1, ±2, ±4, ±8)
    for delta in [1, 2, 4, 8]:
        for sign in [1, -1]:
            still_nan = d['lag'].isna()
            if not still_nan.any():
                break
            tmp = d.loc[still_nan, ['geohash', 'time_slot']].copy()
            tmp['adj'] = tmp['time_slot'] + sign * delta
            adj_df = d48_exact_slot.rename('adj_lag').reset_index()
            adj_df.columns = ['geohash', 'adj', 'adj_lag']
            tmp2 = tmp.merge(adj_df, on=['geohash', 'adj'], how='left')
            filled = tmp2['adj_lag'].notna()
            d.loc[tmp.index[filled], 'lag'] = tmp2.loc[filled, 'adj_lag'].values
    # 3. Geo mean from day48
    still_nan = d['lag'].isna()
    d.loc[still_nan, 'lag'] = d.loc[still_nan, 'geohash'].map(geo_mean_d48)
    # 4. Slot median fallback
    still_nan = d['lag'].isna()
    d.loc[still_nan, 'lag'] = d.loc[still_nan, 'time_slot'].map(slot_med)
    return d['lag'].values


test_df['demand_lag_d1'] = add_lag(test_df)

lag49 = add_lag(train_df[train_df.day == 49].reset_index(drop=True))
train_df['demand_lag_d1'] = np.nan
train_df.loc[train_df.day == 49, 'demand_lag_d1'] = lag49
train_df.loc[train_df.day == 48, 'demand_lag_d1'] = (
    train_df.loc[train_df.day == 48, 'time_slot'].map(slot_med).values)

# Derived lag signals
# Global day ratio (mean day49 / mean day48 per geohash)
day_ratio = (
    train49.groupby('geohash')['demand'].mean() /
    (train48.groupby('geohash')['demand'].mean() + 1e-9)
).clip(0.1, 5.0)

# 2 AM ratio: d49[2:00] / d48[2:00] — per-geohash morning trend
d49_2am = train49[train49.timestamp == '2:0'].groupby('geohash')['demand'].mean()
d48_2am = train48[train48.timestamp == '2:0'].groupby('geohash')['demand'].mean()
geo_2am_ratio = (d49_2am / (d48_2am + 1e-9)).clip(0.1, 10.0)

# Day49 early-data aggregates (available for all test rows)
d49_agg   = train49.groupby('geohash')['demand'].agg(
    d49_mean='mean', d49_max='max', d49_last='last')
geo_stats = train48.groupby('geohash')['demand'].agg(
    geo_mean='mean', geo_max='max', geo_std='std')

# Geo5 spatial-neighbour aggregates from day48
geo5_slot_mean = train48.groupby(['geo5', 'time_slot'])['demand'].mean()
geo5_mean_map  = train48.groupby('geo5')['demand'].mean()

for df in [train_df, test_df]:
    df['day_ratio']     = df['geohash'].map(day_ratio).fillna(1.0)
    df['lag_adjusted']  = df['demand_lag_d1'] * df['day_ratio']
    df['geo_2am_ratio'] = df['geohash'].map(geo_2am_ratio).fillna(1.0)
    df['lag_interp']    = df['demand_lag_d1'] * df['geo_2am_ratio']

    for col in ['d49_mean', 'd49_max', 'd49_last']:
        df[col] = df['geohash'].map(d49_agg[col]).fillna(df['demand_lag_d1'])

    df['geo_mean'] = df['geohash'].map(geo_stats['geo_mean']).fillna(df['demand_lag_d1'])
    df['geo_max']  = df['geohash'].map(geo_stats['geo_max']).fillna(df['demand_lag_d1'])
    df['geo_std']  = df['geohash'].map(geo_stats['geo_std']).fillna(0)
    df['lag_norm'] = df['demand_lag_d1'] / (df['geo_max'] + 1e-9)

    # Geo5 spatial neighbour (vectorised merge)
    geo5_slot_df = geo5_slot_mean.rename('geo5_slot_demand').reset_index()
    df_merged = df[['geo5', 'time_slot']].merge(
        geo5_slot_df, on=['geo5', 'time_slot'], how='left')
    df['geo5_slot_demand'] = (df_merged['geo5_slot_demand']
                              .fillna(df['geo5'].map(geo5_mean_map))
                              .fillna(df['demand_lag_d1']).values)
    df['geo5_mean_demand'] = df['geo5'].map(geo5_mean_map).fillna(df['demand_lag_d1'])

print("  Lag features done.")

# ─────────────────────────────────────────────────────────────
# 5. K-FOLD TARGET ENCODING
# ─────────────────────────────────────────────────────────────
print("K-Fold target encoding...")


def kfold_te(tr, te, col, target='demand', folds=5):
    res_tr = np.zeros(len(tr))
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    for ti, vi in kf.split(tr):
        mp = tr.iloc[ti].groupby(col)[target].mean().to_dict()
        res_tr[vi] = tr[col].iloc[vi].astype(str).map(mp)
    gmp = tr.groupby(col)[target].mean().to_dict()
    om  = tr[target].mean()
    return (pd.Series(res_tr).fillna(om).values,
            pd.Series(te[col].astype(str).map(gmp)).fillna(om).values)


te_cols = ['geohash', 'geo3', 'geo4', 'geo5', 'geo_slot', 'geo_hour', 'geo5_slot',
           'RoadType', 'Weather', 'road_hour', 'road_slot']
for col in te_cols:
    train_df[f'{col}_te'], test_df[f'{col}_te'] = kfold_te(train_df, test_df, col)

# ─────────────────────────────────────────────────────────────
# 6. FEATURE SET
# ─────────────────────────────────────────────────────────────
feats = (
    ['latitude', 'longitude',
     'hour', 'minute', 'time_slot', 'is_weekend', 'is_rush', 'is_night',
     'slot_sin', 'slot_cos', 'hour_sin', 'hour_cos',
     'NumberofLanes', 'Temperature', 'lanes_x_slot',
     'demand_lag_d1', 'lag_adjusted', 'lag_interp', 'day_ratio', 'geo_2am_ratio',
     'd49_mean', 'd49_max', 'd49_last',
     'geo_mean', 'geo_max', 'geo_std', 'lag_norm',
     'geo5_slot_demand', 'geo5_mean_demand'] +
    [f'{c}_te' for c in te_cols]
)
print(f"Total features: {len(feats)}")

for col in feats:
    for df in [train_df, test_df]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

X      = train_df[feats].values
y      = train_df['demand'].values
X_test = test_df[feats].values

# ─────────────────────────────────────────────────────────────
# 7. MODEL A — 3-seed HistGradientBoosting
# ─────────────────────────────────────────────────────────────
print("\nTraining HGBT ensemble (3 seeds × 5 folds)...")

hgbt_params = dict(
    max_iter=600, learning_rate=0.03, max_leaf_nodes=63,
    max_depth=8, min_samples_leaf=10, l2_regularization=0.1,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=30,
)

hgbt_preds = np.zeros(len(X_test))
hgbt_oof   = np.zeros(len(X))

for seed in [42, 2024, 888]:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(X))
    for fold, (ti, vi) in enumerate(kf.split(X, y)):
        m = HistGradientBoostingRegressor(**hgbt_params, random_state=seed)
        m.fit(X[ti], y[ti])
        oof_seed[vi]  = m.predict(X[vi])
        hgbt_preds   += m.predict(X_test) / (5 * 3)
        print(f"  HGBT Seed {seed} | Fold {fold+1} R2: {r2_score(y[vi], oof_seed[vi]):.4f}")
    r2s = r2_score(y, oof_seed)
    print(f"  >> Seed {seed} OOF R2: {r2s:.4f}\n")
    hgbt_oof += oof_seed / 3

print(f"HGBT combined OOF R2: {r2_score(y, hgbt_oof):.4f}")

# ─────────────────────────────────────────────────────────────
# 8. MODEL B — ExtraTrees seed 2024
#    (seed 42 ET got weight 0.00 in OOF search, omitted)
# ─────────────────────────────────────────────────────────────
print("\nTraining ExtraTrees (seed 2024, 5 folds)...")

et_params = dict(n_estimators=200, max_features=0.5,
                 min_samples_leaf=5, n_jobs=-1)

et_preds = np.zeros(len(X_test))
et_oof   = np.zeros(len(X))

kf = KFold(n_splits=5, shuffle=True, random_state=2024)
for fold, (ti, vi) in enumerate(kf.split(X, y)):
    m = ExtraTreesRegressor(**et_params, random_state=2024)
    m.fit(X[ti], y[ti])
    et_oof[vi]  = m.predict(X[vi])
    et_preds   += m.predict(X_test) / 5
    print(f"  ET Fold {fold+1} R2: {r2_score(y[vi], et_oof[vi]):.4f}")

print(f"ET OOF R2: {r2_score(y, et_oof):.4f}")

# ─────────────────────────────────────────────────────────────
# 9. BLEND  (weights from OOF grid search)
#    HGBT=0.55, ET_seed2024=0.45
# ─────────────────────────────────────────────────────────────
# Verify on OOF (optional — uncomment to re-search)
# best_r2, best_w = 0, 0.55
# for w in np.arange(0.4, 1.01, 0.05):
#     r2 = r2_score(y, w*hgbt_oof + (1-w)*et_oof)
#     if r2 > best_r2:
#         best_r2, best_w = r2, w
# print(f"Best blend weight: HGBT={best_w:.2f}, OOF R2={best_r2:.4f}")

W_HGBT = 0.55
W_ET   = 0.45
final_preds = W_HGBT * hgbt_preds + W_ET * et_preds
blend_oof   = W_HGBT * hgbt_oof   + W_ET * et_oof

print(f"\n{'='*45}")
print(f"FINAL BLEND OOF R2:  {r2_score(y, blend_oof):.4f}")
print(f"100 x R2:            {100*r2_score(y, blend_oof):.2f}")
print(f"{'='*45}")

# ─────────────────────────────────────────────────────────────
# 10. SUBMISSION
# ─────────────────────────────────────────────────────────────
final_preds = np.clip(final_preds, 0, None)
sub = pd.DataFrame({'Index': test_df['Index'], 'demand': final_preds})
sub.sort_values('Index').reset_index(drop=True).to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}  Shape: {sub.shape}")

Loading data...
Feature engineering...
Building lag features...
K-Fold target encoding (Mean + Std)...
Denoising lag features...
Features: 38
Applying Quantile Transformer to target...

Training on Un-skewed Target...
Inverse transforming predictions back to 0-1 scale...
Saved! Shape: (41778, 2)
